# 4. 검색 및 답변
Azure AI Search 하이브리드 검색 + GPT-4o-mini 답변 생성

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential
from sentence_transformers import CrossEncoder

load_dotenv()

SEARCH_ENDPOINT = os.getenv('AZURE_SEARCH_ENDPOINT')
SEARCH_API_KEY = os.getenv('AZURE_SEARCH_API_KEY')
SEARCH_INDEX_NAME = os.getenv('AZURE_SEARCH_INDEX_NAME')
OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
EMBEDDING_DEPLOYMENT = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
CHAT_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')
TARGET_YEAR = 2023

openai_client = OpenAI(base_url=OPENAI_ENDPOINT + '/openai/v1', api_key=OPENAI_API_KEY)
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=SEARCH_INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_API_KEY)
)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print('환경변수 로드 완료')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11168.93it/s]


환경변수 로드 완료


## 하이브리드 검색 함수

In [4]:
def hybrid_search(query: str, company: str, top_k: int = 20) -> list[dict]:
    query_embedding = openai_client.embeddings.create(
        input=query,
        model=EMBEDDING_DEPLOYMENT
    ).data[0].embedding

    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=top_k,
        fields='embedding'
    )

    if any(k in query for k in ['리스크', '위험', '위협', '불확실']):
        section_filter = "section eq '위험관리'"
    else:
        section_filter = "section eq '사업의 내용'"

    filter_expr = f"company eq '{company}' and {section_filter}"

    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        filter=filter_expr,
        select=['id', 'company', 'section', 'text', 'chunk_id'],
        top=top_k
    )

    return [
        {
            'id': r['id'],
            'company': r['company'],
            'section': r.get('section', ''),
            'text': r['text'],
            'score': r['@search.score'],
        }
        for r in results
    ]

print('함수 정의 완료')

함수 정의 완료


## Reranking

In [5]:
def rerank(query: str, chunks: list[dict], top_k: int = 5) -> list[dict]:
    pairs = [(query, c['text']) for c in chunks]
    scores = reranker.predict(pairs)
    for i, c in enumerate(chunks):
        c['rerank_score'] = float(scores[i])
    return sorted(chunks, key=lambda x: x['rerank_score'], reverse=True)[:top_k]

print('함수 정의 완료')

함수 정의 완료


## RAG 답변 생성

In [6]:
def rag_answer(query: str, company: str) -> tuple[str, list[dict]]:
    # 1. 하이브리드 검색 (top 20)
    chunks = hybrid_search(query, company=company, top_k=20)

    if not chunks:
        return '관련 정보를 찾을 수 없습니다.', []

    # 2. Reranking (top 5)
    chunks = rerank(query, chunks, top_k=5)

    # 3. 컨텍스트 구성
    context = '\n\n'.join([
        f"[{c['company']} - {c['section']}]\n{c['text']}"
        for c in chunks
    ])

    # 4. GPT-4o-mini 답변 생성
    system_prompt = """당신은 한국 기업의 사업보고서를 분석하는 전문가입니다.
주어진 컨텍스트를 바탕으로 질문에 정확하고 간결하게 답변하세요.
컨텍스트에 재무 수치나 표 데이터가 섞여 있어도 질문과 관련된 핵심 내용만 추출해서 답변하세요.
컨텍스트에 없는 내용은 '보고서에서 확인되지 않습니다'라고 답하세요."""

    response = openai_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'컨텍스트:\n{context}\n\n질문: {query}'},
        ],
        temperature=0,
    )

    return response.choices[0].message.content, chunks

print('함수 정의 완료')

함수 정의 완료


## Reranking 전후 비교

In [14]:
query = '삼성전자의 DS 부문 사업 현황은?'
company = '삼성전자'

chunks_before = hybrid_search(query, company=company, top_k=20)
chunks_after = rerank(query, chunks_before.copy(), top_k=5)

print('=== Reranking 전 (hybrid search score 기준) ===')
for i, c in enumerate(chunks_before[:5], 1):
    print(f'{i}. score={c["score"]:.4f} | {c["text"][:80]}...')

print('\n=== Reranking 후 (rerank score 기준) ===')
for i, c in enumerate(chunks_after, 1):
    print(f'{i}. rerank_score={c["rerank_score"]:.4f} | {c["text"][:80]}...')

=== Reranking 전 (hybrid search score 기준) ===
1. score=0.0331 | 사업의 내용 1. 사업의 개요 당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총괄의 생산ㆍ...
2. score=0.0323 | 사업의 내용 당사는 제품의 특성에 따라 DX(Device eXperience), DS(Device Solutions) 2개의 부문과 패널 사업을...
3. score=0.0318 | 입니다. 가동률은 가동 가능시간 (가동일 ×생산라인 수 ×24시간) 대비 실제 가동 시간으로 산출하였습니다. (단위 : 시간) 부 문 품 목 제...
4. score=0.0313 | et 제품 생산법인, SCS(Xian) 등 반도체 생산법인을 포함하여 총 30개의 법인이 운영되고 있습니다. 2022년 당사의 매출은 302조 ...
5. score=0.0311 | hina Company Limited Hong Kong PT. CHEIL WORLDWIDE INDONESIA Indonesia Cheil Phi...

=== Reranking 후 (rerank score 기준) ===
1. rerank_score=8.6614 | 사업의 내용 당사는 제품의 특성에 따라 DX(Device eXperience), DS(Device Solutions) 2개의 부문과 패널 사업을...
2. rerank_score=8.4856 | 사업의 내용 1. 사업의 개요 당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총괄의 생산ㆍ...
3. rerank_score=8.4386 | 입니다. 가동률은 가동 가능시간 (가동일 ×생산라인 수 ×24시간) 대비 실제 가동 시간으로 산출하였습니다. (단위 : 시간) 부 문 품 목 제...
4. rerank_score=8.0165 | et 제품 생산법인, SCS(Xian) 등 반도체 생산법인을 포함하여 총 30개의 법인이 운영되고

## 테스트 질문 모음

In [ ]:
test_cases = [
    ('삼성전자의 주요 사업 리스크가 뭐야?', '삼성전자'),
    ('SK하이닉스의 HBM 관련 내용을 요약해줘', 'SK하이닉스'),
    ('NAVER의 주요 매출 구조는?', 'NAVER'),
    ('현대자동차의 전기차 전략은?', '현대자동차'),
    ('카카오의 위험 요인은?', '카카오'),
]

for query, company in test_cases:
    print(f'\n질문: {query}')
    print(f'기업: {company}')
    print('-' * 50)
    answer = rag_answer(query, company=company)
    print(answer)
    print()

## 청킹 전략별 답변 품질 비교

In [ ]:
# 같은 질문에 대해 두 전략의 검색 결과 비교
# (3_index.ipynb를 fixed / section 각각 실행 후 인덱스명 바꿔서 비교)

test_query = '삼성전자의 주요 사업 리스크가 뭐야?'

results_hybrid = hybrid_search(test_query, company='삼성전자', top_k=3)

print(f'쿼리: {test_query}')
print(f'\n=== 검색된 청크 ===')
for i, r in enumerate(results_hybrid):
    print(f'\n[{i+1}] 섹션: {r["section"]} | 점수: {r["score"]:.4f}')
    print(r['text'][:200] + '...')